# Milestone 4 Instructions — Eden (eddeness)

## Your Responsibilities
1. **Task 3: RAG Finance Glossary (Option C)** — Create a finance glossary knowledge base and connect it to querychat
2. **Task 7: CHANGELOG (partial)** — Write Release Highlight, Collaboration, and Reflection sections for `[0.4.0]`

**Key files you own:** `data/knowledge_base/finance_glossary.txt`, `src/pages/ai_explorer.py` (RAG integration only), `CHANGELOG.md` (3 sections)

> **No blocking dependencies.** Your work touches different files from other teammates, so you can start immediately.

---

## Part 1: RAG Finance Glossary (Task 3)

### Background

Users viewing the fin-health dashboard encounter financial metrics (ROE, EBITDA, Current Ratio, etc.) that may be unfamiliar. The RAG knowledge base provides a glossary of financial term definitions so that when users ask querychat questions like "What does ROE mean?" or "Explain current ratio", the LLM retrieves and cites the glossary entry rather than generating definitions from memory.

**How it works:**
- `querychat` does **not** have native RAG (no vector DB, no embeddings).
- Instead, we inject the entire glossary into `extra_instructions` as a `<finance_glossary>` XML block.
- The LLM sees the glossary in its system prompt and is instructed to cite it when answering metric-related questions.
- This is a "context-stuffing" RAG approach — simpler than embedding-based RAG but effective for a ~250-line glossary.

---

### Step 1: Create the feature branch

```bash
git checkout develop
git pull origin develop
git checkout -b feat/rag-glossary
git push origin feat/rag-glossary
```

---

### Step 2: Create the knowledge base directory and glossary file

Create a new directory `data/knowledge_base/` and a file `finance_glossary.txt` inside it.

```bash
mkdir -p data/knowledge_base
```

Create `data/knowledge_base/finance_glossary.txt` with definitions for **every** financial metric in the dataset. The file should cover:

**Profitability Metrics:**
- Net Profit Margin
- ROE (Return on Equity)
- ROA (Return on Assets)
- ROI (Return on Investment)

**Income Statement Metrics:**
- Revenue
- Net Income
- Gross Profit
- EBITDA
- Earnings Per Share (EPS)

**Balance Sheet & Liquidity:**
- Current Ratio
- Debt/Equity Ratio
- Shareholder Equity

**Cash Flow Metrics:**
- Cash Flow from Operating Activities
- Cash Flow from Investing Activities
- Cash Flow from Financial Activities
- Free Cash Flow per Share

**Market & Other:**
- Market Capitalisation
- Return on Tangible Equity
- Number of Employees
- Inflation Rate (US)

**Sector Definitions** (BANK, ELEC, FINANCE, FINTECH, FOOD, IT, LOGI, MANUFACTURING)

**How to Interpret Financial Health** (summary checklist)

Each entry must include:
- **Definition** — what the metric measures
- **Formula** — how it is calculated (where applicable)
- **Interpretation** — what high/low values indicate about company health
- **Healthy range** — typical values for a healthy company
- **Dataset range** — approximate min/max values in *our* dataset

---

### Step 3: Here is what the glossary file should look like

Create `data/knowledge_base/finance_glossary.txt` with the following content:

In [ ]:
# Run this cell to see the complete glossary file content
from pathlib import Path

glossary_path = Path("../data/knowledge_base/finance_glossary.txt")
if glossary_path.exists():
    print(glossary_path.read_text(encoding="utf-8"))
else:
    print(f"File not found: {glossary_path.resolve()}")
    print("Create it using the content described in Step 2.")

The full glossary file is approximately 250 lines and covers all metrics listed above. Here is a **condensed example** showing the structure (the real file has all entries):

```text
# Finance Glossary — fin-health Dashboard Knowledge Base

This glossary defines the financial terms and metrics present in the
fin-health dataset.  Use these definitions when users ask about the
meaning, calculation, or interpretation of any metric.

---

## Profitability Metrics

### Net Profit Margin
- **Definition:** The percentage of revenue remaining after all expenses,
  taxes, and costs have been deducted.
- **Formula:** Net Income / Revenue × 100
- **Interpretation:** Higher margins indicate better cost control and
  pricing power.  A negative margin means the company is losing money.
- **Healthy range:** 10 %–20 % for most industries; tech companies often
  exceed 20 %.
- **Dataset range:** approximately −50 % to +35 %.

### Return on Equity (ROE)
- **Definition:** Measures how effectively a company uses shareholder
  equity to generate profit.
- **Formula:** Net Income / Shareholder Equity × 100
- **Interpretation:** Higher ROE signals efficient use of equity capital.
  Very high ROE (> 50 %) may indicate high leverage rather than
  operational excellence.
- **Healthy range:** 15 %–25 %.
- **Dataset range:** approximately −80 % to +160 %.

... (continue for all metrics listed above) ...

## Sector Definitions

| Sector | Tickers | Description |
|--------|---------|-------------|
| BANK | AIG, BCS | Banking and insurance |
| ELEC | INTC, NVDA | Semiconductors and electronics |
| ... | ... | ... |

## How to Interpret Financial Health

A financially healthy company typically exhibits:
1. **Positive and growing revenue** — the business is expanding.
2. **Positive net profit margin** — the company keeps profit after costs.
3. **ROE between 15 %–25 %** — efficient use of shareholder capital.
4. **Current ratio > 1.0** — can meet short-term obligations.
5. **Debt/Equity ratio < 2.0** — not over-leveraged.
6. **Positive operating cash flow** — core operations generate cash.
7. **Positive free cash flow** — surplus cash after capital investments.
```

---

### Step 4: Modify `src/pages/ai_explorer.py` — Add the glossary path constant

Add a `Path` import and a `GLOSSARY_PATH` constant near the top of the file.

**Add `from pathlib import Path` to the imports (line 6):**

```python
import html
import os
from functools import cache
from pathlib import Path       # <-- ADD THIS
```

**Add the glossary path constant after the existing imports (after line 25):**

```python
from components.empty_chart import empty_chart
from data import METRIC_CHOICES, df

GLOSSARY_PATH = (                       # <-- ADD THIS BLOCK
    Path(__file__).parent.parent.parent
    / "data"
    / "knowledge_base"
    / "finance_glossary.txt"
)
```

---

### Step 5: Modify `src/pages/ai_explorer.py` — Add rule 7 to EXTRA_INSTRUCTIONS

Add a new rule at the end of the `EXTRA_INSTRUCTIONS` string that tells the LLM to consult the glossary for financial term questions.

**Find the end of `EXTRA_INSTRUCTIONS` (after rule 6) and add rule 7:**

```python
6. **Never include raw HTML, SQL code blocks, or `<button>` markup in your
   response text.** Do not echo the SQL query or the button element back to the
   user. Just call the appropriate tool and provide the structured summary.

7. **Financial term questions:** When the user asks what a metric means or how
   to interpret a value, consult the <finance_glossary> below and cite the
   definition, formula, and healthy range. Always ground your explanation in
   the glossary rather than generating definitions from memory.
"""
```

---

### Step 6: Modify `src/pages/ai_explorer.py` — Add glossary loading functions

Add two new functions right after the `EXTRA_INSTRUCTIONS` string constant:

```python
def _load_glossary() -> str:
    """Load the finance glossary knowledge base for RAG context."""
    if GLOSSARY_PATH.exists():
        return GLOSSARY_PATH.read_text(encoding="utf-8")
    return ""


def _build_extra_instructions() -> str:
    """Combine base instructions with the finance glossary knowledge base."""
    glossary = _load_glossary()
    if glossary:
        return (
            EXTRA_INSTRUCTIONS
            + "\n<finance_glossary>\n"
            + glossary
            + "\n</finance_glossary>\n"
        )
    return EXTRA_INSTRUCTIONS
```

**What this does:**
- `_load_glossary()` reads the glossary file from disk. If the file doesn't exist, returns an empty string (graceful fallback).
- `_build_extra_instructions()` concatenates the base `EXTRA_INSTRUCTIONS` with the glossary wrapped in `<finance_glossary>` XML tags. This way the LLM sees the glossary in its system prompt.

---

### Step 7: Modify `src/pages/ai_explorer.py` — Update `_get_qc()` to use the combined instructions

In the `_get_qc()` function, change `extra_instructions=EXTRA_INSTRUCTIONS` to `extra_instructions=_build_extra_instructions()`.

**Before:**
```python
@cache
def _get_qc():
    """Lazily create the QueryChat instance (deferred until first use)."""
    return querychat.QueryChat(
        df,
        "financial_data",
        data_description=DATA_DESCRIPTION,
        extra_instructions=EXTRA_INSTRUCTIONS,          # <-- OLD
        greeting=GREETING,
        client=ChatGithub(model="gpt-4.1-mini"),
    )
```

**After:**
```python
@cache
def _get_qc():
    """Lazily create the QueryChat instance (deferred until first use)."""
    return querychat.QueryChat(
        df,
        "financial_data",
        data_description=DATA_DESCRIPTION,
        extra_instructions=_build_extra_instructions(),  # <-- NEW
        greeting=GREETING,
        client=ChatGithub(model="gpt-4.1-mini"),
    )
```

---

### Step 8: Verify the complete modified `ai_explorer.py`

After all changes, the relevant sections of `src/pages/ai_explorer.py` should look like this (showing only the changed/new parts):

In [ ]:
# This cell shows the key changes — DO NOT run this as-is.
# It's a reference for what the file should contain after editing.

# === TOP OF FILE (lines 1-32) ===
# New import on line 6:
from pathlib import Path

# New constant after line 25:
GLOSSARY_PATH = (
    Path(__file__).parent.parent.parent
    / "data"
    / "knowledge_base"
    / "finance_glossary.txt"
)


# === NEW RULE 7 in EXTRA_INSTRUCTIONS (line ~205-209) ===
# Added at the end of the EXTRA_INSTRUCTIONS string, before the closing triple-quote:
"""
7. **Financial term questions:** When the user asks what a metric means or how
   to interpret a value, consult the <finance_glossary> below and cite the
   definition, formula, and healthy range. Always ground your explanation in
   the glossary rather than generating definitions from memory.
"""


# === NEW FUNCTIONS (lines ~212-229) ===
def _load_glossary() -> str:
    """Load the finance glossary knowledge base for RAG context."""
    if GLOSSARY_PATH.exists():
        return GLOSSARY_PATH.read_text(encoding="utf-8")
    return ""


def _build_extra_instructions() -> str:
    """Combine base instructions with the finance glossary knowledge base."""
    glossary = _load_glossary()
    if glossary:
        return (
            EXTRA_INSTRUCTIONS
            + "\n<finance_glossary>\n"
            + glossary
            + "\n</finance_glossary>\n"
        )
    return EXTRA_INSTRUCTIONS


# === UPDATED _get_qc() (line ~239) ===
@cache
def _get_qc():
    """Lazily create the QueryChat instance (deferred until first use)."""
    return querychat.QueryChat(
        df,
        "financial_data",
        data_description=DATA_DESCRIPTION,
        extra_instructions=_build_extra_instructions(),  # CHANGED
        greeting=GREETING,
        client=ChatGithub(model="gpt-4.1-mini"),
    )

### Step 9: Test locally

```bash
shiny run src/app.py
```

**Verify:**
- [ ] App starts without errors
- [ ] Navigate to the "fin-chat" tab
- [ ] Ask: **"What does ROE mean?"** — the response should cite the glossary definition, formula, healthy range (15%–25%), and dataset range (−80% to +160%)
- [ ] Ask: **"What is a healthy current ratio?"** — should explain > 1.0 is healthy, < 1.0 is risky
- [ ] Ask: **"Show IT companies"** — normal filtering should still work as before
- [ ] Ask: **"Explain EBITDA and show the top 3 companies by EBITDA in 2023"** — should give a glossary-grounded definition AND query the data

**The key test for RAG improvement:**

Without the glossary, asking "What does current ratio below 1 mean?" would get a generic LLM response. With the glossary, the response should specifically cite:
- Formula: Current Assets / Current Liabilities
- \> 1.0 = healthy; < 1.0 = potential liquidity risk
- Dataset range: 0.5–4.0

This demonstrates that the RAG context visibly improves the response quality.

```bash
ruff check src/ tests/ --fix && ruff format src/ tests/
pytest tests/ -v
```

---

### Step 10: Create GitHub Issue for Option C

Create a GitHub Issue documenting the option choice and motivation.

**Title:** `M4 Advanced Feature: Option C — RAG Finance Glossary`

**Body:**
```markdown
## Option Choice

We chose **Option C: RAG-based contextual help** for our advanced feature.

## Motivation

The fin-health dashboard presents financial metrics (ROE, EBITDA, Current Ratio,
Debt/Equity Ratio, etc.) that may be unfamiliar to users without a finance background.
A domain-specific knowledge base allows the querychat LLM to provide accurate,
grounded definitions instead of relying on its general training data — reducing the
risk of hallucinated or imprecise explanations.

## Implementation

- Created `data/knowledge_base/finance_glossary.txt` (~250 lines) covering all
  metrics in the dataset with definitions, formulas, healthy ranges, and dataset ranges.
- Integrated the glossary into querychat's `extra_instructions` as a `<finance_glossary>`
  XML block, so the LLM sees it in the system prompt and is instructed to cite it.
- Added rule 7 to `EXTRA_INSTRUCTIONS` directing the LLM to consult the glossary
  for metric-related questions.

## Demonstration

**Query:** "What does current ratio below 1 mean?"

**Without glossary:** Generic response about current ratio.
**With glossary:** Response cites the formula (Current Assets / Current Liabilities),
explains > 1.0 is healthy, < 1.0 is liquidity risk, and notes the dataset range is 0.5–4.0.
```

---

### Step 11: Commit and create PR

```bash
git add data/knowledge_base/finance_glossary.txt src/pages/ai_explorer.py
git commit -m "feat: add RAG finance glossary knowledge base for querychat"
git push origin feat/rag-glossary
```

Create PR: Base `develop` ← Compare `feat/rag-glossary`

**Title:** `feat: Add RAG finance glossary for querychat (Option C)`

**Description:**
```markdown
Fixes #<issue-number>

### Proposed Changes
- Created `data/knowledge_base/finance_glossary.txt` with definitions for all
  financial terms in the dataset (profitability metrics, income statement metrics,
  balance sheet metrics, cash flow metrics, market metrics, sector definitions)
- Added `_load_glossary()` and `_build_extra_instructions()` to `ai_explorer.py`
  to inject the glossary into querychat's system prompt
- Added rule 7 to EXTRA_INSTRUCTIONS directing the LLM to cite glossary entries
- Updated `_get_qc()` to use `_build_extra_instructions()` instead of raw
  `EXTRA_INSTRUCTIONS`

### Testing
- Verified glossary-grounded responses for metric definition queries
- Normal data filtering still works
- `pytest tests/ -v` passes
```

Get review, then merge.

```bash
git checkout develop
git pull origin develop
git branch -D feat/rag-glossary
```

---

## Part 2: CHANGELOG — Release Highlight, Collaboration, Reflection (Task 7)

You are responsible for writing 3 of the 7 subheadings in the `[0.4.0]` CHANGELOG entry. Luke writes the other 4 (Added, Changed, Fixed, Known Issues).

> **Wait until all feature PRs are merged before starting this.** You need to know what was actually shipped.

### Step 1: Create the feature branch

Coordinate with Luke — you both work on the same branch.

```bash
git checkout develop
git pull origin develop
git checkout -b docs/changelog-v0.4.0
git push origin docs/changelog-v0.4.0
```

---

### Step 2: Add your sections to `CHANGELOG.md`

Open `CHANGELOG.md` and add a new `## [v0.4.0]` entry **above** the existing `## [v0.3.0]` entry. Luke will write Added/Changed/Fixed/Known Issues. You write the following three sections.

**Your sections (add after Luke's Known Issues section):**

```markdown
### Release Highlight

**RAG Finance Glossary (Option C)** — We chose Option C (RAG-based contextual help)
to make the fin-chat page more useful for users unfamiliar with financial terminology.
A ~250-line glossary knowledge base (`data/knowledge_base/finance_glossary.txt`) was
created covering all metrics in the dataset with definitions, formulas, healthy ranges,
and interpretation guidance. The glossary is injected into querychat's system prompt so
the LLM cites authoritative definitions rather than generating from memory. For example,
asking "What does current ratio below 1 mean?" now returns a response grounded in the
glossary with the formula, health thresholds, and dataset-specific value ranges.

- **Option choice:** C — RAG-based contextual help
- **Motivation:** Users without finance backgrounds need clear, accurate metric
  explanations when exploring the dashboard. A domain glossary ensures consistent,
  citation-backed answers.
- **PR:** #<rag-pr-number>
- **Issue:** #<option-c-issue-number>

### Collaboration

| Team Member | Primary Contributions |
|-------------|----------------------|
| @jiroamato  | Spec updates, environment setup, parquet + DuckDB migration, deployment |
| @eddeness   | RAG finance glossary (Option C), CHANGELOG (highlight/collaboration/reflection) |
| @ShrutiSasi | Spec + CONTRIBUTING updates, playwright behavior tests |
| @lukeni777  | Unit tests + function refactor, CHANGELOG (added/changed/fixed/known issues) |

All team members addressed at least one feedback item. Work was parallelised across
separate files to minimise merge conflicts: Jiro touched `data.py` and page filtering;
Eden touched `ai_explorer.py` and the knowledge base; Shruti wrote playwright tests;
Luke refactored functions and wrote unit tests.

### Reflection

Milestone 4 focused on production readiness: performance optimisation via DuckDB,
domain-specific knowledge augmentation, and comprehensive test coverage. The parquet
migration with ibis expressions pushes filtering to the database layer, reducing memory
usage for larger datasets. The RAG glossary demonstrates how injecting structured domain
knowledge into an LLM's context can produce more accurate and trustworthy responses
than relying on general training data alone. Adding playwright and pytest tests provides
confidence that the dashboard behaves correctly across filter combinations and page
navigations. These improvements collectively transform the dashboard from a working
prototype into a more robust, well-tested, and user-friendly product.
```

> **Important:** Replace `#<rag-pr-number>` and `#<option-c-issue-number>` with the actual PR and Issue numbers once they exist.

---

### Step 3: Verify the complete CHANGELOG structure

After both you and Luke have added your sections, the `[0.4.0]` entry should have all 7 subheadings:

```
## [v0.4.0] - (2026-03-XX)
├── ### Added           ← Luke
├── ### Changed         ← Luke
├── ### Fixed           ← Luke
├── ### Known Issues    ← Luke
├── ### Release Highlight  ← Eden (you)
├── ### Collaboration      ← Eden (you)
└── ### Reflection         ← Eden (you)
```

---

### Step 4: Commit and create PR

```bash
git add CHANGELOG.md
git commit -m "docs: add release highlight, collaboration, and reflection to CHANGELOG v0.4.0"
git push origin docs/changelog-v0.4.0
```

Create PR (or add to existing if Luke already created one): Base `develop` ← Compare `docs/changelog-v0.4.0`

**Title:** `docs: CHANGELOG v0.4.0`

**Description:**
```markdown
### Proposed Changes
- Added Release Highlight section describing RAG glossary (Option C)
- Added Collaboration table with team member contributions
- Added Reflection section on M4 objectives and outcomes
- All 7 subheadings present: Added / Changed / Fixed / Known Issues /
  Release Highlight / Collaboration / Reflection
```

Get review, merge.

```bash
git checkout develop
git pull origin develop
git branch -D docs/changelog-v0.4.0
```

---

## Review Teammate PRs

You'll be reviewing PRs from other team members when requested.

**When reviewing:**
- Pull the branch locally and run `shiny run src/app.py`
- Verify new functionality works
- Run: `ruff check src/` and `pytest tests/ -v`
- Check that the fin-chat tab still works with the glossary
- Provide constructive feedback

## Summary Checklist

### RAG Glossary (Task 3)
- [ ] `data/knowledge_base/finance_glossary.txt` created with all metric definitions
- [ ] `src/pages/ai_explorer.py` updated with `GLOSSARY_PATH`, `_load_glossary()`, `_build_extra_instructions()`
- [ ] Rule 7 added to `EXTRA_INSTRUCTIONS`
- [ ] `_get_qc()` uses `_build_extra_instructions()` instead of `EXTRA_INSTRUCTIONS`
- [ ] Querychat cites glossary definitions when asked about metrics
- [ ] At least one query demonstrates RAG context improving the response
- [ ] GitHub Issue created documenting Option C choice and motivation
- [ ] PR merged

### CHANGELOG (Task 7)
- [ ] Release Highlight section written (with Option C description, PR link, issue link)
- [ ] Collaboration section written (team member contribution table)
- [ ] Reflection section written
- [ ] All 7 subheadings present in `[0.4.0]` entry
- [ ] PR merged